In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import cv2
from skimage.feature import local_binary_pattern
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report)
import warnings
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")

In [ ]:
class AlzheimerPreprocessor:
    """Advanced preprocessing pipeline for MRI images"""
    
    def __init__(self, target_size=(224, 224)):
        self.target_size = target_size
        
    def apply_gaussian_blur(self, img, ksize=(5, 5)):
        """Apply Gaussian blur to reduce noise"""
        return cv2.GaussianBlur(img, ksize, 0)
    
    def apply_morphology(self, img):
        """Apply erosion and dilation for noise removal"""
        kernel = np.ones((3, 3), np.uint8)
        eroded = cv2.erode(img, kernel, iterations=1)
        dilated = cv2.dilate(eroded, kernel, iterations=1)
        return dilated
    
    def extract_lbp(self, img, radius=1, n_points=8):
        """Extract Local Binary Pattern features"""
        lbp = local_binary_pattern(img, n_points, radius, method='uniform')
        lbp = np.uint8(lbp / lbp.max() * 255)  # Normalize to 0-255
        return lbp
    
    def preprocess(self, img_path):
        """Complete preprocessing pipeline"""
        # Read image in grayscale
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        if img is None:
            raise ValueError(f"Cannot read image: {img_path}")
        
        # Apply Gaussian blur
        img = self.apply_gaussian_blur(img)
        
        # Apply morphological operations
        img = self.apply_morphology(img)
        
        # Extract LBP features
        img_lbp = self.extract_lbp(img)
        
        # Combine original and LBP (weighted average)
        img_combined = cv2.addWeighted(img, 0.7, img_lbp, 0.3, 0)
        
        # Resize to target size
        img_resized = cv2.resize(img_combined, self.target_size, 
                                interpolation=cv2.INTER_CUBIC)
        
        # Normalize to [0, 1]
        img_normalized = img_resized.astype(np.float32) / 255.0
        
        return img_normalized

In [ ]:
class AlzheimerDataset(Dataset):
    """Custom PyTorch Dataset for Alzheimer's MRI"""
    
    def __init__(self, image_paths, labels, preprocessor, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.preprocessor = preprocessor
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        # Preprocess image
        img = self.preprocessor.preprocess(img_path)
        
        # Add channel dimension (grayscale)
        img = np.expand_dims(img, axis=0)
        
        # Convert to tensor
        img = torch.from_numpy(img).float()
        
        # Apply additional transforms if provided
        if self.transform:
            img = self.transform(img)
        
        return img, label


In [ ]:
class HybridResidualBlock(nn.Module):
    """Hybrid Residual Block with optional skip connection"""
    
    def __init__(self, in_channels, out_channels, use_residual=True):
        super(HybridResidualBlock, self).__init__()
        self.use_residual = use_residual
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                              padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, 
                              padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Projection shortcut if dimensions change
        self.shortcut = nn.Sequential()
        if in_channels != out_channels and use_residual:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        identity = x
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.lrelu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        if self.use_residual:
            out += self.shortcut(identity)
        
        out = self.lrelu(out)
        return out


In [ ]:
class LightweightAttention(nn.Module):
    """Custom lightweight attention module"""
    
    def __init__(self, in_channels, reduction_ratio=8):
        super(LightweightAttention, self).__init__()
        
        # Global Average Pooling
        self.gap = nn.AdaptiveAvgPool2d(1)
        
        # Squeeze and Excitation
        self.fc1 = nn.Linear(in_channels, in_channels // reduction_ratio)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(in_channels // reduction_ratio, in_channels)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        b, c, _, _ = x.size()
        
        # Squeeze: Global spatial information
        y = self.gap(x).view(b, c)
        
        # Excitation: Channel attention
        y = self.fc1(y)
        y = self.relu(y)
        y = self.fc2(y)
        y = self.sigmoid(y)
        
        # Scale features
        y = y.view(b, c, 1, 1)
        return x * y.expand_as(x)

In [ ]:
class ShallowMRINet(nn.Module):
    """
    ShallowMRI-Inspired CNN with Hybrid Residual & Attention
    Architecture:
    - 4 Convolutional blocks with optional residual connections
    - Custom attention modules
    - Classification head with batch normalization
    """
    
    def __init__(self, num_classes=4, use_residual=True, use_attention=True):
        super(ShallowMRINet, self).__init__()
        
        self.use_attention = use_attention
        
        # Block 1: Initial feature extraction (no residual)
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.2)
        )
        
        # Block 2: Hybrid residual block
        self.block2 = nn.Sequential(
            HybridResidualBlock(32, 64, use_residual=use_residual),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.3)
        )
        if use_attention:
            self.attn2 = LightweightAttention(64)
        
        # Block 3: Hybrid residual block
        self.block3 = nn.Sequential(
            HybridResidualBlock(64, 128, use_residual=use_residual),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.3)
        )
        if use_attention:
            self.attn3 = LightweightAttention(128)
        
        # Block 4: Hybrid residual block
        self.block4 = nn.Sequential(
            HybridResidualBlock(128, 256, use_residual=use_residual),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.4)
        )
        if use_attention:
            self.attn4 = LightweightAttention(256)
        
        # Classification head
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(256 * 14 * 14, 512)
        self.bn_fc = nn.BatchNorm1d(512)
        self.lrelu_fc = nn.LeakyReLU(0.2, inplace=True)
        self.dropout_fc = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, num_classes)
    
    def forward(self, x):
        # Feature extraction with optional attention
        x = self.block1(x)
        
        x = self.block2(x)
        if self.use_attention:
            x = self.attn2(x)
        
        x = self.block3(x)
        if self.use_attention:
            x = self.attn3(x)
        
        x = self.block4(x)
        if self.use_attention:
            x = self.attn4(x)
        
        # Classification
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.bn_fc(x)
        x = self.lrelu_fc(x)
        x = self.dropout_fc(x)
        x = self.fc2(x)
        
        return x

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc='Training')
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': loss.item(), 
                         'acc': 100. * correct / total})
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


def validate(model, dataloader, criterion, device):
    """Validate model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc='Validation'):
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc, all_preds, all_labels

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, 
                scheduler, num_epochs, device):
    """Complete training loop"""
    best_val_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 
               'val_loss': [], 'val_acc': []}
    
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 60)
        
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, 
                                           optimizer, device)
        
        # Validate
        val_loss, val_acc, _, _ = validate(model, val_loader, criterion, device)
        
        # Update scheduler
        scheduler.step(val_loss)
        
        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
        print(f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_shallowmri_model.pth')
            print(f'✓ Saved best model (Val Acc: {val_acc:.2f}%)')
    
    return history


In [ ]:
def plot_training_history(history):
    """Plot training curves"""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss
    axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
    axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training & Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
    axes[1].plot(history['val_acc'], label='Val Acc', marker='s')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Training & Validation Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
def evaluate_model(model, dataloader, device, class_names):
    """Comprehensive model evaluation"""
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc='Evaluating'):
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')
    
    print("\n" + "="*60)
    print("MODEL EVALUATION METRICS")
    print("="*60)
    print(f"Accuracy:  {accuracy*100:.2f}%")
    print(f"Precision: {precision*100:.2f}%")
    print(f"Recall:    {recall*100:.2f}%")
    print(f"F1-Score:  {f1*100:.2f}%")
    print("="*60)
    
    # Classification report
    print("\nDetailed Classification Report:")
    print(classification_report(all_labels, all_preds, 
                               target_names=class_names))
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return accuracy, precision, recall, f1


In [ ]:
class GradCAM:
    """Gradient-weighted Class Activation Mapping"""
    
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_backward_hook(self.save_gradient)
    
    def save_activation(self, module, input, output):
        self.activations = output.detach()
    
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    
    def generate_cam(self, input_image, target_class):
        """Generate CAM for target class"""
        # Forward pass
        output = self.model(input_image)
        
        # Backward pass
        self.model.zero_grad()
        target = output[0, target_class]
        target.backward()
        
        # Calculate weights
        weights = torch.mean(self.gradients, dim=[2, 3], keepdim=True)
        
        # Generate CAM
        cam = torch.sum(weights * self.activations, dim=1, keepdim=True)
        cam = F.relu(cam)
        
        # Normalize
        cam = cam - cam.min()
        cam = cam / cam.max()
        
        return cam.squeeze().cpu().numpy()


In [ ]:
def visualize_gradcam(model, dataloader, device, class_names, num_samples=4):
    """Visualize Grad-CAM for sample predictions"""
    model.eval()
    
    # Get target layer (last conv block)
    target_layer = model.block4[0].conv2
    gradcam = GradCAM(model, target_layer)
    
    # Get sample images
    images, labels = next(iter(dataloader))
    images, labels = images[:num_samples].to(device), labels[:num_samples]
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 4*num_samples))
    
    for i in range(num_samples):
        img = images[i:i+1]
        true_label = labels[i].item()
        
        # Get prediction
        with torch.no_grad():
            output = model(img)
            pred_label = output.argmax(dim=1).item()
        
        # Generate CAM
        cam = gradcam.generate_cam(img, pred_label)
        
        # Original image
        orig_img = img.squeeze().cpu().numpy()
        axes[i, 0].imshow(orig_img, cmap='gray')
        axes[i, 0].set_title(f'Original\nTrue: {class_names[true_label]}')
        axes[i, 0].axis('off')
        
        # Grad-CAM
        cam_resized = cv2.resize(cam, (224, 224))
        axes[i, 1].imshow(cam_resized, cmap='jet')
        axes[i, 1].set_title(f'Grad-CAM\nPred: {class_names[pred_label]}')
        axes[i, 1].axis('off')
        
        # Overlay
        overlay = orig_img.copy()
        cam_colored = cv2.applyColorMap(np.uint8(255 * cam_resized), 
                                       cv2.COLORMAP_JET)
        cam_colored = cv2.cvtColor(cam_colored, cv2.COLOR_BGR2RGB) / 255.0
        overlay = overlay[..., np.newaxis] * 0.5 + cam_colored[:, :, 0] * 0.5
        axes[i, 2].imshow(overlay, cmap='gray')
        axes[i, 2].set_title('Overlay')
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig('gradcam_visualization.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
def main():
    """Main execution pipeline"""
    
    # Configuration
    DATA_PATH = '/kaggle/input/augmented-alzheimer-mri-dataset/AugmentedAlzheimerDataset'
    BATCH_SIZE = 32
    NUM_EPOCHS = 30
    LEARNING_RATE = 0.001
    NUM_CLASSES = 4
    CLASS_NAMES = ['MildDemented', 'ModerateDemented', 
                   'NonDemented', 'VeryMildDemented']
    
    print("="*60)
    print("SHALLOWMRI-INSPIRED CNN FOR ALZHEIMER'S CLASSIFICATION")
    print("="*60)
    
    # 1. Load dataset paths
    print("\n[1/7] Loading dataset...")
    image_paths = []
    labels = []
    
    for idx, class_name in enumerate(CLASS_NAMES):
        class_path = os.path.join(DATA_PATH, class_name)
        if os.path.exists(class_path):
            for img_name in os.listdir(class_path):
                img_path = os.path.join(class_path, img_name)
                if img_path.endswith(('.jpg', '.jpeg', '.png')):
                    image_paths.append(img_path)
                    labels.append(idx)
    
    print(f"Total images: {len(image_paths)}")
    for idx, name in enumerate(CLASS_NAMES):
        count = labels.count(idx)
        print(f"  {name}: {count}")
    
    # 2. Split dataset
    print("\n[2/7] Splitting dataset...")
    X_train, X_temp, y_train, y_temp = train_test_split(
        image_paths, labels, test_size=0.3, random_state=42, stratify=labels
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )
    
    print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
    
    # 3. Create datasets and dataloaders
    print("\n[3/7] Creating data loaders...")
    preprocessor = AlzheimerPreprocessor(target_size=(224, 224))
    
    train_dataset = AlzheimerDataset(X_train, y_train, preprocessor)
    val_dataset = AlzheimerDataset(X_val, y_val, preprocessor)
    test_dataset = AlzheimerDataset(X_test, y_test, preprocessor)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, 
                             shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, 
                           shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, 
                            shuffle=False, num_workers=2, pin_memory=True)
    
    # 4. Initialize model
    print("\n[4/7] Initializing model...")
    model = ShallowMRINet(num_classes=NUM_CLASSES, 
                         use_residual=True, 
                         use_attention=True)
    model = model.to(device)
    
    # Multi-GPU support
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    
    # 5. Training setup
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, verbose=True
    )
    
    # 6. Train model
    print("\n[5/7] Training model...")
    history = train_model(model, train_loader, val_loader, criterion, 
                         optimizer, scheduler, NUM_EPOCHS, device)
    
    # Plot training history
    plot_training_history(history)
    
    # 7. Load best model and evaluate
    print("\n[6/7] Evaluating on test set...")
    if torch.cuda.device_count() > 1:
        model.module.load_state_dict(torch.load('best_shallowmri_model.pth'))
    else:
        model.load_state_dict(torch.load('best_shallowmri_model.pth'))
    
    accuracy, precision, recall, f1 = evaluate_model(
        model, test_loader, device, CLASS_NAMES
    )
    
    # 8. Grad-CAM visualization
    print("\n[7/7] Generating Grad-CAM visualizations...")
    visualize_gradcam(model, test_loader, device, CLASS_NAMES, num_samples=4)
    
    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print("="*60)
    print(f"Best Model Saved: best_shallowmri_model.pth")
    print(f"Final Test Accuracy: {accuracy*100:.2f}%")
    print("="*60)


if __name__ == '__main__':
    main()